# Chapter 16 — Agriculture and Rural India

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch16_agriculture_rural_india.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 16.1 Two files, one household

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from indiadata import read_plfs, layout_names, mospi_nas

PLFS = Path("data/plfs")
KEY = ["qtr", "visit", "sec", "st", "dc", "strm", "sstrm", "ss", "mfsu",
       "seg", "sss", "ssu"]

h = read_plfs("hhv1", folder=PLFS,
              columns=["dc", "seg", "sss", "ssu", "hh_size", "hhtype",
                       "hce2", "hce_tot"])
p = read_plfs("perv1", folder=PLFS,
              columns=["dc", "seg", "sss", "ssu", "age", "sex", "pas", "sas",
                       "ind_pas", "ind_sas"])
print(f"households {len(h):,}  persons {len(p):,}  "
      f"memory {(h.memory_usage(deep=True).sum() + p.memory_usage(deep=True).sum()) / 1e6:.0f} MB")
print("duplicate household keys:", h.duplicated(KEY).sum())

In [ ]:
size = p.groupby(KEY, observed=True).size().rename("persons_in_file").reset_index()
check = h[KEY + ["hh_size"]].merge(size, on=KEY, how="outer",
                                   indicator=True, validate="1:1")
print(check["_merge"].value_counts().to_string())
print("hh_size equals number of person records:",
      (check["hh_size"] == check["persons_in_file"]).mean())
del check, size

## 16.2 From persons to households

In [ ]:
layout_file = next(PLFS.glob("Data_Layout*.xlsx"))
states = pd.read_excel(layout_file, sheet_name="State code", header=1)
states.columns = ["st", "state"]
states = states.dropna().astype({"st": int})
CROSSWALK = {"Tamilnadu": "Tamil Nadu", "Puduchery": "Puducherry",
             "Andaman & N. Island": "Andaman & Nicobar Islands"}
STATE = {k: CROSSWALK.get(v.strip(), v.strip())
         for k, v in zip(states["st"], states["state"])}

works = p["pas"].between(11, 51) | p["sas"].fillna(0).between(11, 51)
nic = p["ind_pas"].where(p["pas"].between(11, 51), p["ind_sas"])
division = nic // 1000                 # <NA> where there is no industry code
p["worker"] = works & (p["age"] >= 15)
p["farm"] = p["worker"] & division.le(3).fillna(False)
p["nonfarm"] = p["worker"] & division.gt(3).fillna(False)
print("workers with no industry code:", int((p["worker"] & nic.isna()).sum()))

w = p["weight"]
emp_state = pd.DataFrame({
    "agri share of workers %": 100 * w[p["farm"]].groupby(p["st"]).sum()
                               / w[p["worker"]].groupby(p["st"]).sum(),
    "sample workers": p[p["worker"]].groupby("st").size()})
emp_state.index = emp_state.index.map(STATE)
india_emp = 100 * w[p["farm"]].sum() / w[p["worker"]].sum()
print(f"All India: agriculture's share of workers {india_emp:.1f}%")

In [ ]:
rural_p = p[p["sec"] == 1]
rw = rural_p[rural_p["worker"] & rural_p["sex"].isin([1, 2])]
share_nf = lambda d: 100 * d.loc[d["nonfarm"], "weight"].sum() / d["weight"].sum()
nonfarm_share = rw.groupby(["st", "sex"]).apply(share_nf).unstack("sex")
nonfarm_share.columns = ["Men", "Women"]
nonfarm_share["rural sample workers"] = rw.groupby("st").size()
nonfarm_share.index = nonfarm_share.index.map(STATE)
india_nf = rw.groupby("sex").apply(share_nf)
print(f"Rural workers in non-farm work: men {india_nf[1]:.1f}%, "
      f"women {india_nf[2]:.1f}%")

In [ ]:
hh_work = rural_p.groupby(KEY, observed=True).agg(
    workers=("worker", "sum"), farm_workers=("farm", "sum"),
    nonfarm_workers=("nonfarm", "sum")).reset_index()
del p, rural_p, rw, works, nic, division, w

r = h[h["sec"] == 1].merge(hh_work, on=KEY, how="left", validate="1:1")
del h, hh_work
r["mpce"] = r["hce_tot"].astype(float) / r["hh_size"].astype(float)
r["pweight"] = r["weight"] * r["hh_size"]
r["w_sub"] = r["mult"] / 100 / r["no_qtr"] * r["hh_size"]
print(f"{len(r):,} rural households")
r[["hhtype", "hh_size", "workers", "farm_workers", "nonfarm_workers", "mpce"]].head()

## 16.3 Are agricultural households poorer?

In [ ]:
HHTYPE = {1: "Self-employed in agriculture", 2: "Self-employed in non-agriculture",
          3: "Regular wage/salary", 4: "Casual labour in agriculture",
          5: "Casual labour in non-agriculture", 9: "Others (no earnings)"}
r["type"] = r["hhtype"].map(HHTYPE)


def wmean(d, col="mpce", w="pweight"):
    return np.average(d[col], weights=d[w])


def subsample_se(d, stat, weight="weight", sub_weight="w_sub"):
    """Statistic on the full sample and on each PLFS sub-sample (ss = 1, 2),
    with the half-difference standard error. stat(d, w) takes a data frame
    and the name of a weight column; it may return a number or a Series."""
    full = stat(d, weight)
    s1 = stat(d[d["ss"] == 1], sub_weight)
    s2 = stat(d[d["ss"] == 2], sub_weight)
    out = {"full sample": full, "sub-sample 1": s1, "sub-sample 2": s2,
           "approx. SE": abs(s1 - s2) / 2}
    return pd.DataFrame(out) if isinstance(full, pd.Series) else pd.Series(out)


def est_se(d, stat, weight="pweight"):
    s = subsample_se(d, stat, weight=weight)
    return pd.Series({"estimate": s["full sample"], "SE": s["approx. SE"]})


mean_mpce = lambda x, w: wmean(x, "mpce", w)
types = r.groupby("type").apply(lambda d: pd.concat([
    pd.Series({"share of rural persons %": 100 * d["pweight"].sum() / r["pweight"].sum(),
               "household size": wmean(d, "hh_size", "weight"),
               "home-grown % of consumption": 100 * (d["hce2"] * d["weight"]).sum()
                                               / (d["hce_tot"] * d["weight"]).sum()}),
    est_se(d, mean_mpce).add_prefix("MPCE ")]))
types = types.sort_values("MPCE estimate")
types.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))
colours = ["#c55a11" if t in (HHTYPE[1], HHTYPE[4]) else "#9db7d5"
           for t in types.index]
ax.barh(types.index, types["MPCE estimate"], xerr=2 * types["MPCE SE"],
        color=colours, error_kw={"lw": 1, "capsize": 3})
for i, (v, se) in enumerate(zip(types["MPCE estimate"], types["MPCE SE"])):
    ax.text(40, i, f"Rs {v:,.0f}", va="center", ha="left",
            fontsize=8.5, color="white" if colours[i] == "#c55a11" else "black")
avg = wmean(r)
ax.axvline(avg, color="black", lw=1, ls="--")
ax.text(avg + 30, len(types) - 0.45, f"rural average Rs {avg:,.0f}", fontsize=8)
ax.set_xlim(0, 3800)
ax.set_ylim(-0.6, len(types) - 0.2)
ax.set_xlabel("Mean MPCE, Rs per person per month")
ax.set_title("Rural MPCE by household type, 2023–24", loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.07, "Orange: agricultural household types. Error bars: ±2 "
         "sub-sample SEs. Person-weighted. Source: author's calculation\nfrom "
         "PLFS 2023–24 household file (NSO, MoSPI).", fontsize=8, color="grey")
plt.show()

In [ ]:
r["farm_type"] = np.select([r["hhtype"].isin([1, 4]), r["hhtype"].isin([2, 3, 5])],
                           ["agricultural", "non-agricultural"], "others")
fa = r[r["farm_type"] != "others"]


def gap(d, w):
    """Mean MPCE of agricultural minus non-agricultural households."""
    m = d.groupby("farm_type").apply(lambda g: np.average(g["mpce"], weights=g[w]))
    return m["agricultural"] - m["non-agricultural"]


raw = pd.concat({ft: est_se(g, mean_mpce)
                 for ft, g in fa.groupby("farm_type")}, axis=1).T
raw.loc["difference"] = est_se(fa, gap)
raw.round(1)

In [ ]:
cell = fa.groupby(["st", "farm_type"]).apply(lambda g: wmean(g)).unstack()
state_pop = fa.groupby("st")["pweight"].sum() / fa["pweight"].sum()
standard = cell.mul(state_pop, axis=0).sum()
print("Mean MPCE if both groups had the all-India rural distribution across states:")
print(standard.round(0).to_string())
print(f"Standardised gap: Rs {standard['agricultural'] - standard['non-agricultural']:.0f}")

In [ ]:
r["mix"] = np.select(
    [r["workers"] == 0, r["nonfarm_workers"] == 0, r["farm_workers"] == 0],
    ["no worker", "farm only", "non-farm only"], "both farm and non-farm")
mix = r.groupby("mix").apply(lambda d: pd.concat([
    pd.Series({"share of rural persons %": 100 * d["pweight"].sum() / r["pweight"].sum()}),
    est_se(d, mean_mpce).add_prefix("MPCE ")]))
mix.round(1)

## 16.4 What does a day of casual work pay?

In [ ]:
names = layout_names("perv1", folder=PLFS)
pd.DataFrame({"field": names[60:70]}).T

In [ ]:
CASUAL = (41, 42, 51)


def read_casual_days(folder=PLFS):
    """Person-day records of casual wage work from Block 6 of perv1.

    Reads the CSV in chunks and keeps only persons with a casual-labour
    status on at least one day, so memory stays small.
    """
    names = layout_names("perv1", folder=folder)
    header = pd.read_csv(folder / "perv1.csv", nrows=0).columns
    rename = dict(zip(header, names))
    daily = [f"{v}{a}{d}" for d in range(1, 8) for a in (1, 2)
             for v in ("das", "ind", "ern")]
    keep = ["qtr", "sec", "st", "ss", "nss", "nsc", "mult", "no_qtr", "sex",
            "age"] + daily
    usecols = [hd for hd, n in rename.items() if n in keep]
    parts = []
    for chunk in pd.read_csv(folder / "perv1.csv", usecols=usecols,
                             chunksize=100_000):
        chunk = chunk.rename(columns=rename)
        status = chunk[[c for c in chunk if c.startswith("das")]]
        parts.append(chunk[status.isin(CASUAL).any(axis=1)])
    wide = pd.concat(parts, ignore_index=True)
    wide["weight"] = np.where(wide["nss"] == wide["nsc"], wide["mult"] / 100,
                              wide["mult"] / 200) / wide["no_qtr"]
    wide["w_sub"] = wide["mult"] / 100 / wide["no_qtr"]
    wide["person"] = wide.index
    fixed = ["person", "qtr", "sec", "st", "ss", "sex", "age", "weight", "w_sub"]
    long = []
    for d in range(1, 8):
        for a in (1, 2):
            one = wide[fixed + [f"das{a}{d}", f"ind{a}{d}", f"ern{a}{d}"]]
            one.columns = fixed + ["status", "nic2", "wage"]
            long.append(one[one["status"].isin(CASUAL)].assign(day=d))
    return pd.concat(long, ignore_index=True)


acts = read_casual_days()
print(f"{len(acts):,} casual-work activity records; "
      f"memory {acts.memory_usage(deep=True).sum() / 1e6:.0f} MB")
acts["status"].value_counts()

In [ ]:
other = acts[acts["status"] == 51]
days = other.groupby(["person", "day"]).agg(
    wage=("wage", "sum"), nic2=("nic2", "first"), qtr=("qtr", "first"),
    sec=("sec", "first"), st=("st", "first"), ss=("ss", "first"),
    sex=("sex", "first"), weight=("weight", "first"),
    w_sub=("w_sub", "first")).reset_index()
print(f"{len(days):,} person-days of casual work other than public works; "
      f"days with two such activities: {(other.groupby(['person', 'day']).size() > 1).sum()}")
print("days with no industry code:", int(days["nic2"].isna().sum()))

In [ ]:
def day_wage(d, w="weight"):
    """Average wage per day worked: weighted total wages / weighted days."""
    return (d["wage"] * d[w]).sum() / d[w].sum()


rural_days = days[days["sec"] == 1]
ours = rural_days.groupby(["qtr", "sex"]).apply(day_wage).unstack()
ours["Person"] = rural_days.groupby("qtr").apply(day_wage)
ours.columns = ["Male", "Female", "Person"]
published = pd.DataFrame([[423, 289, 387], [435, 287, 400], [437, 286, 404],
                          [444, 299, 417]], index=ours.index,
                         columns=["Male", "Female", "Person"])
pd.concat({"ours": ours.round(0), "MoSPI": published,
           "difference": ours.round(0) - published}, axis=1)

In [ ]:
rural_days = rural_days[rural_days["sex"].isin([1, 2])].assign(
    work=np.where(rural_days["nic2"].le(3), "agricultural", "non-agricultural"),
    gender=rural_days["sex"].map({1: "Male", 2: "Female"}))
wages = rural_days.groupby(["gender", "work"]).apply(lambda d: pd.concat([
    est_se(d, day_wage, weight="weight"),
    pd.Series({"share of days %": 100 * d["weight"].sum()
               / rural_days.loc[rural_days["gender"] == d.name[0], "weight"].sum(),
               "sample days": len(d)})])).unstack("work")
nonag = rural_days[rural_days["work"] == "non-agricultural"]
constr = nonag.loc[nonag["nic2"].between(41, 43), "weight"].sum() / nonag["weight"].sum()
print(f"Construction (NIC 41-43): {100 * constr:.1f}% of rural non-farm casual days")
wages.round(1)

In [ ]:
men = rural_days[rural_days["gender"] == "Male"].groupby(["st", "work"])
wst = pd.DataFrame({"wage": men.apply(day_wage), "n": men.size()}).unstack("work")
wst.index = wst.index.map(STATE)
wst = wst[(wst["n"] >= 300).all(axis=1)]["wage"].sort_values("agricultural")
wst["ratio"] = wst["agricultural"] / wst["non-agricultural"]
print(f"{len(wst)} states with at least 300 sample days in each type of work")
print(f"farm wage below non-farm wage in all: {(wst['ratio'] < 1).all()}")
print(f"ratio: {wst['ratio'].min():.2f} ({wst['ratio'].idxmin()}) to "
      f"{wst['ratio'].max():.2f} ({wst['ratio'].idxmax()}); farm wage: "
      f"Rs {wst['agricultural'].min():.0f} ({wst['agricultural'].idxmin()}) to "
      f"Rs {wst['agricultural'].max():.0f} ({wst['agricultural'].idxmax()})")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
y = np.arange(len(wst))
ax.hlines(y, wst["agricultural"], wst["non-agricultural"], color="lightgrey", lw=2)
ax.scatter(wst["agricultural"], y, color="#c55a11", zorder=3, label="Agricultural work")
ax.scatter(wst["non-agricultural"], y, color="#1f4e79", zorder=3,
           label="Non-agricultural work")
ax.set_yticks(y)
ax.set_yticklabels(wst.index, fontsize=8.5)
ax.set_xlabel("Average wage per day worked, Rs")
ax.set_title("Rural casual wages of men, 2023–24", loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
fig.text(0.01, -0.02, "Casual labour other than public works (status 51), "
         "daily records of the reference week, men, rural. States with at least\n"
         "300 sample days in each type of work. Source: author's calculation "
         "from PLFS 2023–24 person file (NSO, MoSPI).", fontsize=8, color="grey")
plt.tight_layout()
plt.show()
del acts, other, days

## 16.5 The productivity gap, state by state

In [ ]:
gsva = mospi_nas("GSVA by industry")
print(gsva["unit"].unique(), gsva["year"].max())
gsva[["year", "state", "industry", "subindustry", "current_price"]].head(6)

In [ ]:
g23 = gsva[(gsva["fy"] == 2023) & gsva["subindustry"].isna()].pivot_table(
    index="state", columns="industry", values="current_price", aggfunc="sum")
gva_state = pd.DataFrame({
    "agri share of GSVA %": 100 * g23["Agriculture, Livestock, Forestry and Fishing"]
                            / g23["Total Gross State Value Added"],
    "GSVA (Rs lakh crore)": g23["Total Gross State Value Added"] / 1e7})
gap_state = emp_state.join(gva_state, how="inner")
print("PLFS states without NAS GSVA:",
      sorted(set(emp_state.index) - set(gva_state.index)))
gap_state["relative productivity"] = (gap_state["agri share of GSVA %"]
                                      / gap_state["agri share of workers %"])
major = gap_state[(gap_state["GSVA (Rs lakh crore)"] >= 2)
                  & (gap_state["agri share of workers %"] >= 5)].sort_values(
    "relative productivity")
major[["agri share of workers %", "agri share of GSVA %",
       "relative productivity"]].round(2)

In [ ]:
nudge = {"Uttarakhand": (-30, -24), "Maharashtra": (14, -16),
         "Karnataka": (9, 2), "Gujarat": (-45, 4), "Telangana": (8, -4),
         "West Bengal": (6, 3), "Jammu & Kashmir": (-60, 30),
         "Odisha": (6, -3)}
arrow = {"Uttarakhand", "Jammu & Kashmir"}
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(major["agri share of workers %"], major["agri share of GSVA %"],
           s=major["GSVA (Rs lakh crore)"] * 12, color="#9db7d5",
           edgecolor="#1f4e79")
for s in major.index:
    ax.annotate(s, (major.loc[s, "agri share of workers %"],
                    major.loc[s, "agri share of GSVA %"]),
                fontsize=7.5, xytext=nudge.get(s, (6, 2)),
                textcoords="offset points",
                arrowprops={"arrowstyle": "-", "color": "grey", "lw": 0.6}
                if s in arrow else None)
ax.plot([0, 70], [0, 70], color="grey", lw=1, ls="--")
ax.text(50, 55, "output share =\nemployment share", fontsize=8, color="grey")
ax.set_xlim(0, 70)
ax.set_ylim(0, 70)
ax.set_aspect("equal")
ax.set_xlabel("Agriculture's share of workers, % (PLFS 2023–24)")
ax.set_ylabel("Agriculture's share of GSVA, % (2023–24)")
ax.set_title("Agriculture's share of output is below its share of jobs\n"
             "in every major state", loc="left", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.02, "Workers: usual status (ps+ss), aged 15+, weighted. GSVA "
         "at current prices, latest estimate. Bubble size: GSVA.\nStates with "
         "GSVA of Rs 2 lakh crore or more, excluding Delhi. Sources: author's "
         "calculation from PLFS 2023–24\nunit-level data; MoSPI NAS.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
rp = major["relative productivity"]
print(f"Relative productivity of agriculture: lowest {rp.idxmin()} ({rp.min():.2f}), "
      f"highest {rp.idxmax()} ({rp.max():.2f}), median {rp.median():.2f}")
print(f"Correlation of employment share and output share across these states: "
      f"{major['agri share of workers %'].corr(major['agri share of GSVA %']):.2f}")

## 16.6 Diversification into rural non-farm work

In [ ]:
nf = nonfarm_share[nonfarm_share["rural sample workers"] >= 3000].sort_values("Men")
print(f"women below men in every state: {(nf['Women'] < nf['Men']).all()}")
print("Highest women's shares:", nf["Women"].nlargest(3).round(1).to_dict())
print(nf[["Men", "Women"]].round(1).agg(["idxmin", "idxmax"]).to_string())
print(nf[["Men", "Women"]].round(1).agg(["min", "max"]).to_string())

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading